# Stage 2 — Plain MAPPO (PAH OFF)

**Config:** 5 drones · moving targets · 5 obstacles · conflict graph ON · PAH OFF

**Why PAH OFF first:** If Stage 2 base fails, we know it's the config (not PAH).
Only after this run confirms convergence → create `kaggle_stage2_pah.ipynb`.

**Changes from Stage 1 (v4):**
| | Stage 1 (v4) | Stage 2 (this) |
|---|---|---|
| Drones | 3 | **5** |
| Obstacles | 0 | **5** |
| Targets | Static | **Moving (1 m/step)** |
| Conflict graph | OFF (obs=10) | **ON (obs=30)** |
| max_steps | 300 | **500** |
| total_episodes | 5000 | **8000** |
| PAH | OFF | **OFF (plain MAPPO)** |

In [ ]:
# Cell 1 — Imports
import torch
import numpy as np
from scipy.optimize import linear_sum_assignment
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device   : {DEVICE}')

In [ ]:
# Cell 2 — Conflict Graph
# Ported from code/algorithms/conflict_graph.py

import numpy as np

def _time_to_cpa(pos_i, pos_j, vel_i, vel_j):
    p = pos_i - pos_j
    v = vel_i - vel_j
    vv = np.dot(v, v)
    if vv < 1e-8:
        return np.inf
    return -np.dot(p, v) / vv

def _dist_at_cpa(pos_i, pos_j, vel_i, vel_j, horizon):
    p = pos_i - pos_j
    v = vel_i - vel_j
    vv = np.dot(v, v)
    if vv < 1e-8:
        return float(np.linalg.norm(p)), 0.0
    t_star = -np.dot(p, v) / vv
    t_clamped = float(np.clip(t_star, 0.0, horizon))
    return float(np.linalg.norm(p + v * t_clamped)), t_star

class ConflictGraph:
    K_NBR = 4

    def __init__(self, n_drones, horizon=3.0, d_danger=9.0):
        self.n      = n_drones
        self.H      = horizon
        self.d_dan  = d_danger
        self.adj    = np.zeros((n_drones, n_drones), dtype=bool)
        self.t_mat  = np.full((n_drones, n_drones), np.inf)

    def update(self, pos, vel):
        self.adj[:] = False
        self.t_mat[:] = np.inf
        for i in range(self.n):
            for j in range(i + 1, self.n):
                dcpa, t_star = _dist_at_cpa(pos[i], pos[j], vel[i], vel[j], self.H)
                self.t_mat[i, j] = self.t_mat[j, i] = t_star
                if dcpa < self.d_dan and 0.0 <= t_star <= self.H:
                    self.adj[i, j] = self.adj[j, i] = True

    def n_conflict(self, i):
        return int(self.adj[i].sum())

    def tau_collision(self, i):
        nbrs = np.where(self.adj[i])[0]
        if len(nbrs) == 0:
            return self.H
        return float(np.min(self.t_mat[i, nbrs]))

    def neighbor_obs(self, i, pos, vel):
        obs = np.zeros(self.K_NBR * 5, dtype=np.float32)
        nbrs = np.where(self.adj[i])[0]
        if len(nbrs) == 0:
            return obs
        order = np.argsort(self.t_mat[i, nbrs])
        nbrs  = nbrs[order]
        for slot, j in enumerate(nbrs[:self.K_NBR]):
            b = slot * 5
            obs[b:b+2]   = pos[j] - pos[i]
            obs[b+2:b+4] = vel[j] - vel[i]
            obs[b+4]     = 1.0
        return obs

In [ ]:
# Cell 3 — Environment
# Changes from v4:
#   - Moving targets: target_vel initialized at reset, targets drift + bounce
#   - Conflict graph ON: obs_dim = 30 (10 base + 4*5 neighbors)
#   - n_drones=5, n_obstacles=5 (set via CFG in Cell 4)
#   - Poisson-disk obstacle placement (same as code/environment/multi_uav_env.py)

import numpy as np
from scipy.optimize import linear_sum_assignment

K_NBR    = 4
BASE_DIM = 10
OBS_DIM  = BASE_DIM + K_NBR * 5  # 30

class MultiUAVEnv:
    def __init__(self, n_drones=5, n_obstacles=5, world_size=500.0,
                 max_speed=5.0, max_steps=500, collision_radius=3.0,
                 target_radius=25.0, success_bonus=20.0,
                 target_speed=1.0, seed=None):

        self.n          = n_drones
        self.n_obs      = n_obstacles
        self.W          = world_size
        self.max_sp     = max_speed
        self.max_steps  = max_steps
        self.col_r      = collision_radius
        self.tgt_r      = target_radius
        self.bonus      = success_bonus
        self.tgt_speed  = target_speed
        self.rng        = np.random.default_rng(seed)
        self.cg         = ConflictGraph(n_drones, horizon=3.0,
                                        d_danger=collision_radius * 3.0)

        self.obs_dim    = OBS_DIM
        self.act_dim    = 2

    def reset(self):
        self.t = 0
        all_pos = self._sample_non_overlapping(self.n * 2, self.col_r * 2)
        self.drone_pos  = all_pos[:self.n].copy()
        self.target_pos = all_pos[self.n:].copy()
        self.obstacle_pos = self._place_obstacles(
            np.concatenate([self.drone_pos, self.target_pos]), self.n_obs
        )
        self.drone_vel  = np.zeros((self.n, 2), dtype=np.float32)

        # Moving targets: random slow velocity for each target
        angles = self.rng.uniform(0, 2 * np.pi, self.n)
        self.target_vel = (self.tgt_speed * np.stack(
            [np.cos(angles), np.sin(angles)], axis=1
        )).astype(np.float32)

        self.assignment = self._hungarian()
        self.cg.update(self.drone_pos, self.drone_vel)
        return self._obs()

    def step(self, actions):
        self.t += 1
        actions = np.clip(actions, -self.max_sp, self.max_sp)
        self.drone_vel = actions.astype(np.float32)
        self.drone_pos = np.clip(self.drone_pos + self.drone_vel, 0.0, self.W)

        # Move targets and bounce off walls
        self.target_pos = self.target_pos + self.target_vel
        for k in range(self.n):
            for dim in range(2):
                if self.target_pos[k, dim] < 10.0:
                    self.target_pos[k, dim] = 10.0
                    self.target_vel[k, dim] *= -1.0
                elif self.target_pos[k, dim] > self.W - 10.0:
                    self.target_pos[k, dim] = self.W - 10.0
                    self.target_vel[k, dim] *= -1.0

        self.assignment = self._hungarian()
        self.cg.update(self.drone_pos, self.drone_vel)

        rewards, r_mission, r_safety = self._rewards()

        all_reached   = self._all_reached()
        any_collision = self._any_collision()
        timeout       = self.t >= self.max_steps
        terminated    = all_reached or any_collision
        truncated     = timeout

        if all_reached and self.bonus:
            r_mission += self.bonus
            rewards   += self.bonus

        info = {
            'all_targets_reached': all_reached,
            'any_collision':       any_collision,
            'r_mission':           r_mission,
            'r_safety':            r_safety,
        }
        return self._obs(), rewards, terminated, truncated, info

    def _obs(self):
        obs = np.zeros((self.n, self.obs_dim), dtype=np.float32)
        for i in range(self.n):
            rel = self.target_pos[self.assignment[i]] - self.drone_pos[i]
            cl  = self._clearances(i)
            base = np.concatenate([
                self.drone_pos[i] / self.W,
                self.drone_vel[i] / self.max_sp,
                rel               / self.W,
                cl,
            ])
            nbr = self.cg.neighbor_obs(i, self.drone_pos, self.drone_vel)
            obs[i] = np.concatenate([base, nbr])
        return obs

    def _rewards(self):
        r_mission = np.zeros(self.n, dtype=np.float32)
        r_safety  = np.zeros(self.n, dtype=np.float32)
        d_danger  = self.col_r * 3.0
        zone_w    = d_danger - self.col_r

        for i in range(self.n):
            rel  = self.target_pos[self.assignment[i]] - self.drone_pos[i]
            dist = np.linalg.norm(rel)
            direction  = rel / (dist + 1e-6)
            r_progress = float(np.dot(self.drone_vel[i], direction) / self.max_sp)
            r_mission[i] = 0.4 * r_progress + 0.3 * (-dist / self.W)

            min_d = float('inf')
            for j in range(self.n):
                if j != i:
                    min_d = min(min_d, np.linalg.norm(self.drone_pos[i] - self.drone_pos[j]))
            for op in self.obstacle_pos:
                min_d = min(min_d, np.linalg.norm(self.drone_pos[i] - op))

            if min_d < self.col_r:
                r_safety[i] = -1.0
            elif min_d < d_danger:
                r_safety[i] = -((d_danger - min_d) / zone_w)

        rewards = r_mission + 0.3 * r_safety
        return rewards, r_mission, r_safety

    def _hungarian(self):
        cost = np.array([
            [np.linalg.norm(self.drone_pos[i] - self.target_pos[j])
             for j in range(self.n)]
            for i in range(self.n)
        ])
        _, col = linear_sum_assignment(cost)
        return col

    def _all_reached(self):
        return all(
            np.linalg.norm(self.drone_pos[i] - self.target_pos[self.assignment[i]]) <= self.tgt_r
            for i in range(self.n)
        )

    def _any_collision(self):
        for i in range(self.n):
            for j in range(self.n):
                if j != i and np.linalg.norm(self.drone_pos[i] - self.drone_pos[j]) < self.col_r:
                    return True
            for op in self.obstacle_pos:
                if np.linalg.norm(self.drone_pos[i] - op) < self.col_r:
                    return True
        return False

    def _clearances(self, i):
        pos = self.drone_pos[i]
        cl  = np.array([self.W - pos[1], pos[1], self.W - pos[0], pos[0]], dtype=np.float32)
        for op in self.obstacle_pos:
            diff = op - pos
            if diff[1] > 0: cl[0] = min(cl[0],  diff[1])
            else:           cl[1] = min(cl[1], -diff[1])
            if diff[0] > 0: cl[2] = min(cl[2],  diff[0])
            else:           cl[3] = min(cl[3], -diff[0])
        return cl / self.W

    def _sample_non_overlapping(self, n, min_dist):
        pts, attempts = [], 0
        while len(pts) < n:
            attempts += 1
            if attempts > 10000:
                raise RuntimeError('World too crowded.')
            c = self.rng.uniform(10.0, self.W - 10.0, 2)
            if not any(np.linalg.norm(c - p) < min_dist for p in pts):
                pts.append(c)
        return np.array(pts, dtype=np.float32)

    def _place_obstacles(self, existing, n_obs, max_tries=1000):
        if n_obs <= 0:
            return np.zeros((0, 2), dtype=np.float32)
        r_min = 2.0 * self.col_r
        placed = []
        for _ in range(n_obs):
            for _ in range(max_tries):
                c = self.rng.uniform(10.0, self.W - 10.0, 2)
                if any(np.linalg.norm(c - o) < r_min for o in placed):
                    continue
                if any(np.linalg.norm(c - p) < self.col_r for p in existing):
                    continue
                placed.append(c)
                break
            else:
                break
        return np.array(placed, dtype=np.float32) if placed else np.zeros((0, 2), dtype=np.float32)

print(f'obs_dim={OBS_DIM}  (10 base + {K_NBR}×5 conflict neighbors)')

In [ ]:
# Cell 4 — MAPPO (Actor + Critic + Buffer)
# Unchanged from v4 — obs_dim passed in at construction time

import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal

class Actor(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),  nn.Tanh(),
        )
        self.mu_head  = nn.Linear(hidden, act_dim)
        self.log_std  = nn.Parameter(torch.zeros(act_dim))

    def forward(self, x):
        h   = self.net(x)
        mu  = self.mu_head(h)
        std = self.log_std.exp().expand_as(mu)
        return Normal(mu, std)

class Critic(nn.Module):
    def __init__(self, obs_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),  nn.Tanh(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

class RolloutBuffer:
    def __init__(self):
        self.obs, self.acts, self.rews = [], [], []
        self.vals, self.lps, self.dones = [], [], []

    def add(self, obs, act, rew, val, lp, done):
        self.obs.append(obs)
        self.acts.append(act)
        self.rews.append(rew)
        self.vals.append(val)
        self.lps.append(lp)
        self.dones.append(done)

    def clear(self):
        self.__init__()

    def compute_gae(self, last_val, gamma, lam):
        T   = len(self.rews)
        adv = np.zeros(T, dtype=np.float32)
        gae = 0.0
        vals_np  = np.array(self.vals,  dtype=np.float32)
        rews_np  = np.array(self.rews,  dtype=np.float32)
        dones_np = np.array(self.dones, dtype=np.float32)
        for t in reversed(range(T)):
            nxt   = last_val if t == T - 1 else vals_np[t + 1]
            delta = rews_np[t] + gamma * nxt * (1 - dones_np[t]) - vals_np[t]
            gae   = delta + gamma * lam * (1 - dones_np[t]) * gae
            adv[t] = gae
        returns = adv + vals_np
        return adv, returns

class MAPPOAgent:
    def __init__(self, obs_dim, act_dim, lr=3e-4, gamma=0.99, lam=0.95,
                 clip_eps=0.2, vf_coef=0.5, ent_coef=0.003,
                 n_epochs=10, batch_size=64):
        self.actor  = Actor(obs_dim, act_dim).to(DEVICE)
        self.critic = Critic(obs_dim).to(DEVICE)
        self.opt    = optim.Adam(
            list(self.actor.parameters()) + list(self.critic.parameters()), lr=lr
        )
        self.gamma, self.lam         = gamma, lam
        self.clip_eps                = clip_eps
        self.vf_coef, self.ent_coef  = vf_coef, ent_coef
        self.n_epochs, self.batch_size = n_epochs, batch_size

    @torch.no_grad()
    def get_actions(self, obs):
        obs_t = torch.tensor(obs, dtype=torch.float32).to(DEVICE)
        dist  = self.actor(obs_t)
        acts  = dist.sample()
        lps   = dist.log_prob(acts).sum(-1)
        vals  = self.critic(obs_t)
        return acts.cpu().numpy(), lps.cpu().numpy(), vals.cpu().numpy()

    def update(self, buffer, last_val):
        adv_np, ret_np = buffer.compute_gae(last_val, self.gamma, self.lam)

        obs_np = np.array(buffer.obs,  dtype=np.float32)
        act_np = np.array(buffer.acts, dtype=np.float32)
        lp_np  = np.array(buffer.lps,  dtype=np.float32)

        T, N, D  = obs_np.shape
        obs_flat = obs_np.reshape(T * N, D)
        act_flat = act_np.reshape(T * N, -1)
        ret_flat = np.repeat(ret_np, N)
        lp_flat  = np.repeat(lp_np, N)   # lp_np shape (T,) → repeat N times → (T*N,)
        adv_flat = np.repeat(adv_np, N)
        adv_flat = (adv_flat - adv_flat.mean()) / (adv_flat.std() + 1e-8)

        a_losses, v_losses, entropies = [], [], []
        for _ in range(self.n_epochs):
            idx = np.random.permutation(len(obs_flat))
            for start in range(0, len(obs_flat), self.batch_size):
                b     = idx[start:start + self.batch_size]
                obs_b = torch.tensor(obs_flat[b]).to(DEVICE)
                act_b = torch.tensor(act_flat[b]).to(DEVICE)
                ret_b = torch.tensor(ret_flat[b]).to(DEVICE)
                lp_b  = torch.tensor(lp_flat[b]).to(DEVICE)
                adv_b = torch.tensor(adv_flat[b]).to(DEVICE)

                dist  = self.actor(obs_b)
                log_p = dist.log_prob(act_b).sum(-1)
                ratio = (log_p - lp_b).exp()
                surr1 = ratio * adv_b
                surr2 = ratio.clamp(1 - self.clip_eps, 1 + self.clip_eps) * adv_b
                a_loss = -torch.min(surr1, surr2).mean()
                v_loss = (self.critic(obs_b) - ret_b).pow(2).mean()
                e_loss = dist.entropy().sum(-1).mean()
                loss   = a_loss + self.vf_coef * v_loss - self.ent_coef * e_loss

                self.opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(
                    list(self.actor.parameters()) + list(self.critic.parameters()), 0.5
                )
                self.opt.step()
                a_losses.append(a_loss.item())
                v_losses.append(v_loss.item())
                entropies.append(e_loss.item())

        return np.mean(a_losses), np.mean(v_losses), np.mean(entropies)

In [ ]:
# Cell 5 — Config
# ONE CHANGE AT A TIME rule: this run tests Stage 2 base (no PAH).
# Do NOT change ent_coef, lr, or reward weights without a separate diagnosed reason.

import os

CFG = {
    # Environment
    'n_drones':        5,
    'n_obstacles':     5,
    'world_size':      500.0,
    'max_speed':       5.0,
    'max_steps':       500,       # more budget: moving targets + obstacles
    'target_radius':   25.0,      # same as Stage 1 curriculum value
    'collision_radius': 3.0,
    'success_bonus':   20.0,
    'target_speed':    1.0,       # NEW: targets drift at 1 m/step
    # Training
    'total_episodes':  8000,
    'rollout_steps':   512,
    'eval_every':      100,
    'eval_episodes':   20,
    'save_every':      500,
    'seed':            42,
    # MAPPO
    'lr':              3e-4,
    'gamma':           0.99,
    'lam':             0.95,
    'clip_eps':        0.2,
    'vf_coef':         0.5,
    'ent_coef':        0.003,     # validated in Stage 1
    'n_epochs':        10,
    'batch_size':      64,
    # Output
    'run_name':        'output-stage-2-plain',
}

RESULTS_DIR = os.path.join(os.getcwd(), CFG['run_name'])
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Results → {RESULTS_DIR}')

np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])

In [ ]:
# Cell 6 — Train

import time, json

def make_env(bonus=CFG['success_bonus']):
    return MultiUAVEnv(
        n_drones         = CFG['n_drones'],
        n_obstacles      = CFG['n_obstacles'],
        world_size       = CFG['world_size'],
        max_speed        = CFG['max_speed'],
        max_steps        = CFG['max_steps'],
        collision_radius = CFG['collision_radius'],
        target_radius    = CFG['target_radius'],
        success_bonus    = bonus,
        target_speed     = CFG['target_speed'],
        seed             = CFG['seed'],
    )

env      = make_env()
eval_env = make_env(bonus=0.0)   # honest eval — no bonus
agent    = MAPPOAgent(
    obs_dim    = OBS_DIM,
    act_dim    = 2,
    lr         = CFG['lr'],
    gamma      = CFG['gamma'],
    lam        = CFG['lam'],
    clip_eps   = CFG['clip_eps'],
    vf_coef    = CFG['vf_coef'],
    ent_coef   = CFG['ent_coef'],
    n_epochs   = CFG['n_epochs'],
    batch_size = CFG['batch_size'],
)

buffer   = RolloutBuffer()
history  = {'episode': [], 'success': [], 'collision': [],
             'actor_loss': [], 'critic_loss': [], 'entropy': []}
obs      = env.reset()
ep_count = 0
start    = time.time()

header = " {:>9} | {:>8} | {:>10} | {:>8} | {:>8} | {:>7}".format(
    "Episode", "Success", "Collision", "A-Loss", "Entropy", "Time"
)
print(header)
print('-' * 65)

while ep_count < CFG['total_episodes']:
    # ---- collect rollout ----
    for _ in range(CFG['rollout_steps']):
        acts, lps, vals = agent.get_actions(obs)
        next_obs, rews, terminated, truncated, info = env.step(acts)
        done = terminated or truncated
        buffer.add(obs, acts, rews.mean(), vals.mean(), lps.mean(), done)
        obs = next_obs
        if done:
            ep_count += 1
            obs = env.reset()

    # ---- update ----
    _, last_vals, _ = agent.get_actions(obs)
    a_loss, v_loss, entropy = agent.update(buffer, last_vals.mean())
    buffer.clear()

    # ---- eval ----
    if ep_count % CFG['eval_every'] == 0:
        successes, collisions = 0, 0
        for _ in range(CFG['eval_episodes']):
            o    = eval_env.reset()
            done = False
            while not done:
                a, _, _ = agent.get_actions(o)
                o, _, terminated, truncated, info = eval_env.step(a)
                done = terminated or truncated
            if info['all_targets_reached']: successes  += 1
            if info['any_collision']:       collisions += 1

        sr      = successes  / CFG['eval_episodes']
        cr      = collisions / CFG['eval_episodes']
        elapsed = (time.time() - start) / 60
        print(" {:>9} | {:>7.1%} | {:>9.1%} | {:>+8.4f} | {:>8.4f} | {:>6.1f}m".format(
            ep_count, sr, cr, a_loss, entropy, elapsed
        ))

        history['episode'].append(ep_count)
        history['success'].append(sr)
        history['collision'].append(cr)
        history['actor_loss'].append(a_loss)
        history['critic_loss'].append(v_loss)
        history['entropy'].append(entropy)

    # ---- checkpoint ----
    if ep_count % CFG['save_every'] == 0:
        ckpt = {'actor': agent.actor.state_dict(), 'critic': agent.critic.state_dict()}
        torch.save(ckpt, "{}/checkpoint_ep{}.pt".format(RESULTS_DIR, ep_count))
        print("  >> checkpoint saved: {}/checkpoint_ep{}.pt".format(RESULTS_DIR, ep_count))

# ---- final save ----
torch.save({'actor': agent.actor.state_dict(), 'critic': agent.critic.state_dict()},
           '{}/final_model.pt'.format(RESULTS_DIR))
with open('{}/history.json'.format(RESULTS_DIR), 'w') as f:
    json.dump(history, f, indent=2)
print('\nDone. final_model.pt saved to {}'.format(RESULTS_DIR))

In [ ]:
# Cell 7 — Results Plot

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(f'Stage 2 Training — {CFG["run_name"]} (PAH OFF, plain MAPPO)', fontweight='bold')

axes[0].plot(history['episode'], [s*100 for s in history['success']], color='green', linewidth=2, label='Success')
axes[0].plot(history['episode'], [c*100 for c in history['collision']], 'r--', linewidth=1.5, label='Collision')
axes[0].set_title('Success vs Collision Rate')
axes[0].set_ylabel('%'); axes[0].set_xlabel('Episode')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history['episode'], history['actor_loss'],  color='blue',   label='Actor Loss')
axes[1].plot(history['episode'], history['critic_loss'], color='orange',  label='Critic Loss')
axes[1].set_title('Training Losses')
axes[1].set_xlabel('Episode')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(history['episode'], history['entropy'], color='purple', linewidth=2)
axes[2].axhline(0, color='gray', linewidth=0.8, linestyle=':')
axes[2].set_title('Policy Entropy')
axes[2].set_xlabel('Episode')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved.')